In [9]:
from pathlib import Path
import sys

# Identify project root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Add project root to Python's module search path
sys.path.insert(0, str(PROJECT_ROOT))

# import utility function
from src.utils import describe_crs

# Provide input to describe CRS function
crs_4326 = describe_crs("EPSG:4326")
crs_32760 = describe_crs("EPSG:32760")

# print the results
print("EPSG:4326")
print("Units:", crs_4326["units"])
print("Datum:", crs_4326["datum"])

print("\nEPSG:32760")
print("Units:", crs_32760["units"])
print("Datum:", crs_32760["datum"])

CRS Name: WGS 84
EPSG Code: EPSG:4326
WKT: GEOGCRS["WGS 84",
    ENSEMBLE["World Geodetic System 1984 ensemble",
        MEMBER["World Geodetic System 1984 (Transit)"],
        MEMBER["World Geodetic System 1984 (G730)"],
        MEMBER["World Geodetic System 1984 (G873)"],
        MEMBER["World Geodetic System 1984 (G1150)"],
        MEMBER["World Geodetic System 1984 (G1674)"],
        MEMBER["World Geodetic System 1984 (G1762)"],
        MEMBER["World Geodetic System 1984 (G2139)"],
        MEMBER["World Geodetic System 1984 (G2296)"],
        ELLIPSOID["WGS 84",6378137,298.257223563,
            LENGTHUNIT["metre",1]],
        ENSEMBLEACCURACY[2.0]],
    PRIMEM["Greenwich",0,
        ANGLEUNIT["degree",0.0174532925199433]],
    CS[ellipsoidal,2],
        AXIS["geodetic latitude (Lat)",north,
            ORDER[1],
            ANGLEUNIT["degree",0.0174532925199433]],
        AXIS["geodetic longitude (Lon)",east,
            ORDER[2],
            ANGLEUNIT["degree",0.0174532925199433]

/Users/robmoore/geoanalytics/geog313-assignments/assignment-5/.pixi/envs/default/lib/python3.14/site-packages/pyproj/crs/crs.py:1295: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj = self._crs.to_proj4(version=version)


In [11]:
# build a fiji region polygon
from shapely.geometry import Polygon

# Fiji region coordinates (longitude, latitude)
coords = [
    (177.2, -16.1),
    (-178.3, -16.1),
    (-178.3, -19.2),
    (177.2, -19.2)
]

# create polygon
fiji_geom = Polygon(coords)

# calculate bounding box and area
print("Bounding box:", fiji_geom.bounds)
print("Area (square degrees):", fiji_geom.area)

Bounding box: (-178.3, -19.2, 177.2, -16.1)
Area (square degrees): 1102.0499999999993


In [12]:
from src.utils import shift_to360

# shuift Fiji polygon to 0-360 longitude convention
fiji_360 = shift_to360(fiji_geom)

# calculate bounding box
print("Original bounding box:", fiji_geom.bounds)
print("Shifted bounding box:", fiji_360.bounds)

# compare longitude widths
original_width = fiji_geom.bounds[2] - fiji_geom.bounds[0]
shifted_width = fiji_360.bounds[2] - fiji_360.bounds[0]

print("Original width:", original_width, "degrees")
print("Shifted width:", shifted_width, "degrees")

Original bounding box: (-178.3, -19.2, 177.2, -16.1)
Shifted bounding box: (177.2, -19.2, 181.7, -16.1)
Original width: 355.5 degrees
Shifted width: 4.5 degrees


In [16]:
# Import utility
from src.utils import reproject_geometry

# Define equal-area projection centered on Fiji
equal_area_crs = (
    "+proj=laea +lat_0=-17.5 +lon_0=180 "
    "+datum=WGS84 +units=m +no_defs"
)

# Reproject the corrected Fiji polygon
fiji_projected = reproject_geometry(
    fiji_360,
    "EPSG:4326",
    equal_area_crs
)

# Calculate area in square kilometers
area_km2 = fiji_projected.area / 1e6

print("Fiji region area:", round(area_km2, 2), "km2")

Fiji region area: 163784.55 km2


In [17]:
from shapely.geometry import box
from src.utils import utm_epsg

# build Taveuni AOI
aoi = box(179.85, -16.95, 179.99, -16.80)

# calculate centroid
centroid = aoi.centroid

lon = centroid.x
lat = centroid.y

print("Centroid longitude:", lon)
print("Centroid latitude:", lat)

# Determine UTM zone
zone, hemisphere, epsg = utm_epsg(lon, lat)

print("UTM Zone:", zone)
print("Hemisphere:", hemisphere)
print("EPSG:", epsg)

Centroid longitude: 179.92000000000002
Centroid latitude: -16.875
UTM Zone: 60
Hemisphere: south
EPSG: 32760


In [19]:
# compare to point across antimeridian
west_point = utm_epsg(179.9, -1)

# Point just east of the antimeridian
east_point = utm_epsg(-179.9, -1)

print("West of antimeridian:", west_point)
print("East of antimeridian:", east_point)

West of antimeridian: (60, 'south', 32760)
East of antimeridian: (1, 'south', 32701)
